In [ ]:
import pandas as pd

df = pd.read_csv("Project2_data_2025.csv")
print(df.shape)
df = df.drop(columns=["Unnamed: 0"])
df = df.dropna()
print(df.shape)
print(df.head())

(1048570, 14)
(1048526, 13)
   ORIG_AMT  ORIG_TRM  NUM_BO  DTI PROP_TYP  reperformance_flag  Loan.Age  \
0        95       360       2   30       SF                   0        10   
1        95       360       1   55       SF                   0        44   
2       242       360       1   50       SF                   0        11   
3        92       360       2   50       SF                   0         3   
4       155       360       2   30       SF                   0        22   

   VinYr  FicoBkt    LAST_VAL  MLTV  Default STATE  
0   2000      780  105.298846    90       -1    IL  
1   2000      740  214.892455    45       -1    IL  
2   2000      700  331.233585    75       -1    CA  
3   2000      780  101.799593    95       -1    OH  
4   2000      780  268.251889    60       -1    MN  


In [ ]:

import numpy as np

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense
from scikeras.wrappers import KerasClassifier



train_df = df[df['VinYr'] == 2000].drop(columns=['STATE', 'VinYr'])
test_df  = df[df['VinYr'] == 2001].drop(columns=['STATE', 'VinYr'])

# 3. Original targets
y_train_orig = train_df['Default']   # values: 0 = default (positive), -1 = non-default (negative)
y_test_orig  = test_df['Default']
X_train_full = train_df.drop(columns=['Default'])
X_test       = test_df.drop(columns=['Default'])

# 4. Recode to {0,1} for classifiers: 1=default, 0=non-default
y_train_full = (y_train_orig == 0).astype(int)
y_test       = (y_test_orig  == 0).astype(int)

In [ ]:
numeric_cols = X_train_full.select_dtypes(include=['number']).columns.tolist()
cat_cols     = X_train_full.select_dtypes(include=['object','category']).columns.tolist()
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
])

idx40k      = X_train_full.sample(n=40000, random_state=42).index
X_sub       = X_train_full.loc[idx40k]
y_sub       = y_train_full.loc[idx40k]        # {0,1}
y_sub_orig  = y_train_orig.loc[idx40k]        # {0,-1}

X_tune, X_val, y_tune, y_val, y_tune_orig, y_val_orig = train_test_split(
    X_sub, y_sub, y_sub_orig,
    test_size=0.2, random_state=42, stratify=y_sub
)

# 6. Calculate scale_pos_weight
# 6a. count how many 0s and 1s in tuning data
n_negative = (y_tune == 0).sum()
n_positive = (y_tune == 1).sum()
scale_pos_weight = n_negative / n_positive

print(f"scale_pos_weight = {scale_pos_weight:.2f}")



scale_pos_weight = 38.65


In [ ]:
# 11. Evaluation on test set with original labels
def evaluate(model, X, y_orig, is_rnn=False):
    if is_rnn:
        X_proc = preprocessor.transform(X).reshape(-1,1,preprocessor.transform(X).shape[1])
        y_bin_pred = (model.predict(X_proc).ravel()>0.5).astype(int)
    else:
        y_bin_pred = model.predict(X)
    # map back to original labels
    y_pred = np.where(y_bin_pred==1, 0, -1)
    cm = confusion_matrix(y_orig, y_pred, labels=[-1,0])
    tn, fp, fn, tp = cm.ravel()
    print(f"\n=== {model.steps[-1][0]} ===")
    print("Confusion matrix (true [-1,0] vs pred [-1,0]):\n", cm)
    print("Accuracy:      ", (tp+tn)/(tp+tn+fp+fn))
    print("Precision(0):  ", tp/(tp+fp) if tp+fp>0 else 0)
    print("Recall(0):     ", tp/(tp+fn) if tp+fn>0 else 0)
    print("Specificity:   ", tn/(tn+fp) if tn+fp>0 else 0)
    print("F1(0):         ", 2*tp/(2*tp+fp+fn) if 2*tp+fp+fn>0 else 0)

In [ ]:

# 7. Helper to get F1 on validation
def f1_val(pipeline):
    y_bin_pred = pipeline.predict(X_val)

    return f1_score(y_val, y_bin_pred, pos_label=1)


# 8a. XGBoost manual grid search
pipe_xgb = Pipeline([
    ('pre', preprocessor),
    ('clf', XGBClassifier(
        eval_metric='logloss', random_state=42, scale_pos_weight=scale_pos_weight
    ))
])

best_score, best_params_xgb = -1, {}
results_xgb = []

for n in [1500, 1000, 500, 200]:
    for d in [20, 10,5]:
        for lr in [0.05, 0.1, 0.2]:
            pipe_xgb.set_params(
                clf__n_estimators=n,
                clf__max_depth=d,
                clf__learning_rate=lr
            )
            pipe_xgb.fit(X_tune, y_tune)

            print(f"\nTrying: n_estimators={n}, max_depth={d}, learning_rate={lr}")
            evaluate(pipe_xgb, X_val, y_val_orig)

            score = f1_val(pipe_xgb)
            results_xgb.append({
                'n_estimators': n,
                'max_depth': d,
                'learning_rate': lr,
                'f1_val': score
            })

            if score > best_score:
                best_score = score
                best_params_xgb = {'n_estimators': n, 'max_depth': d, 'learning_rate': lr}

print("\n=== XGB Grid Search Completed ===")
print("Best params based on F1 on validation set:", best_params_xgb)


Trying: n_estimators=1500, max_depth=20, learning_rate=0.05

=== clf ===
Confusion matrix (true [-1,0] vs pred [-1,0]):
 [[7744   55]
 [  95  106]]
Accuracy:       0.98125
Precision(0):   0.6583850931677019
Recall(0):      0.527363184079602
Specificity:    0.9929478138222849
F1(0):          0.585635359116022

Trying: n_estimators=1500, max_depth=20, learning_rate=0.1

=== clf ===
Confusion matrix (true [-1,0] vs pred [-1,0]):
 [[7746   53]
 [  96  105]]
Accuracy:       0.981375
Precision(0):   0.6645569620253164
Recall(0):      0.5223880597014925
Specificity:    0.99320425695602
F1(0):          0.584958217270195

Trying: n_estimators=1500, max_depth=20, learning_rate=0.2

=== clf ===
Confusion matrix (true [-1,0] vs pred [-1,0]):
 [[7742   57]
 [  98  103]]
Accuracy:       0.980625
Precision(0):   0.64375
Recall(0):      0.5124378109452736
Specificity:    0.9926913706885498
F1(0):          0.5706371191135734

Trying: n_estimators=1500, max_depth=10, learning_rate=0.05

=== clf ===
Con

In [ ]:
df_results = pd.DataFrame(results_xgb)
print(df_results.sort_values('f1_val', ascending=False))

    n_estimators  max_depth  learning_rate    f1_val
9           1000         20           0.05  0.587258
19           500         20           0.10  0.587258
0           1500         20           0.05  0.585635
29           200         20           0.20  0.584958
1           1500         20           0.10  0.584958
28           200         20           0.10  0.584699
18           500         20           0.05  0.584699
5           1500         10           0.20  0.583569
10          1000         20           0.10  0.583333
11          1000         20           0.20  0.577778
23           500         10           0.20  0.576271
20           500         20           0.20  0.576177
14          1000         10           0.20  0.575499
32           200         10           0.20  0.573816
4           1500         10           0.10  0.573816
22           500         10           0.10  0.572207
27           200         20           0.05  0.571429
17          1000          5           0.20  0.

In [ ]:
best_params_xgb = {'n_estimators': 500, 'max_depth': 10, 'learning_rate': 0.2}
print("\n=== XGB Grid Search Completed ===")
print("Best params based on confusion matrix on validation set:", best_params_xgb)


=== XGB Grid Search Completed ===
Best params based on confusion matrix on validation set: {'n_estimators': 500, 'max_depth': 10, 'learning_rate': 0.2}


In [ ]:
# 1. Set up the MLP pipeline
pipe_mlp = Pipeline([
    ('pre', preprocessor),
    ('clf', MLPClassifier(
        random_state=42,
        max_iter=240,
        tol=1e-4,
        verbose=False
    ))
])

best_score = -1
best_params_mlp = {}
results_mlp = []

for hidden_layers in [(10,), (20,), (20, 10), (30, 20, 10)]:
    for alpha in [1e-4, 1e-3]:
        for lr in [5e-3, 1e-2, 1e-1]:
            pipe_mlp.set_params(
                clf__hidden_layer_sizes=hidden_layers,
                clf__alpha=alpha,
                clf__learning_rate_init=lr,
                clf__learning_rate='constant'
            )


            pipe_mlp.fit(X_tune, y_tune)

            print(f"\nTrying: layers={hidden_layers}, alpha={alpha}, lr_init={lr}")
            evaluate(pipe_mlp, X_val, y_val_orig)


            score = f1_val(pipe_mlp)
            results_mlp.append({
                'hidden_layers': hidden_layers,
                'alpha': alpha,
                'learning_rate_init': lr,
                'f1_val': score
            })

            if score > best_score:
                best_score = score
                best_params_mlp = {
                    'hidden_layer_sizes': hidden_layers,
                    'alpha': alpha,
                    'learning_rate_init': lr
                }

print("\n=== MLP Grid Search Completed ===")
print("Best params based on F1 on validation set:", best_params_mlp)


Trying: layers=(10,), alpha=0.0001, lr_init=0.005

=== clf ===
Confusion matrix (true [-1,0] vs pred [-1,0]):
 [[7773   25]
 [  98  104]]
Accuracy:       0.984625
Precision(0):   0.8062015503875969
Recall(0):      0.5148514851485149
Specificity:    0.9967940497563478
F1(0):          0.6283987915407855

Trying: layers=(10,), alpha=0.0001, lr_init=0.01

=== clf ===
Confusion matrix (true [-1,0] vs pred [-1,0]):
 [[7776   22]
 [  96  106]]
Accuracy:       0.98525
Precision(0):   0.828125
Recall(0):      0.5247524752475248
Specificity:    0.997178763785586
F1(0):          0.6424242424242425

Trying: layers=(10,), alpha=0.0001, lr_init=0.1

=== clf ===
Confusion matrix (true [-1,0] vs pred [-1,0]):
 [[7760   38]
 [  92  110]]
Accuracy:       0.98375
Precision(0):   0.7432432432432432
Recall(0):      0.5445544554455446
Specificity:    0.9951269556296486
F1(0):          0.6285714285714286

Trying: layers=(10,), alpha=0.001, lr_init=0.005

=== clf ===
Confusion matrix (true [-1,0] vs pred [-1

In [ ]:
df_results_mlp = pd.DataFrame(results_mlp)
print(df_results_mlp.sort_values('f1_val', ascending=False))

   hidden_layers   alpha  learning_rate_init    f1_val
10         (20,)  0.0010               0.010  0.652695
22  (30, 20, 10)  0.0010               0.010  0.650456
6          (20,)  0.0001               0.005  0.650456
12      (20, 10)  0.0001               0.005  0.648148
4          (10,)  0.0010               0.010  0.648148
7          (20,)  0.0001               0.010  0.646526
1          (10,)  0.0001               0.010  0.642424
8          (20,)  0.0001               0.100  0.640244
21  (30, 20, 10)  0.0010               0.005  0.637427
13      (20, 10)  0.0001               0.010  0.634731
19  (30, 20, 10)  0.0001               0.010  0.633721
9          (20,)  0.0010               0.005  0.632911
5          (10,)  0.0010               0.100  0.631902
2          (10,)  0.0001               0.100  0.628571
0          (10,)  0.0001               0.005  0.628399
3          (10,)  0.0010               0.005  0.628399
23  (30, 20, 10)  0.0010               0.100  0.623229
18  (30, 2

In [ ]:
print("\n=== MLP Grid Search Completed ===")
print("Best params based on F1 on validation set:", best_params_mlp)


=== MLP Grid Search Completed ===
Best params based on F1 on validation set: {'hidden_layer_sizes': (20,), 'alpha': 0.001, 'learning_rate_init': 0.01}


In [ ]:
import random
from sklearn.preprocessing import FunctionTransformer
from scikeras.wrappers       import KerasClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, SimpleRNN, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import confusion_matrix, f1_score

X_tune_proc = preprocessor.fit_transform(X_tune)
X_val_proc  = preprocessor.transform(X_val)
n_features  = X_tune_proc.shape[1]

reshape = lambda X: X.reshape(-1, 1, X.shape[1])

def build_rnn_model(units=32, dropout_rate=0.2, learning_rate=1e-3):
    m = Sequential([
        Input(shape=(1, n_features)),
        SimpleRNN(units, dropout=dropout_rate),
        Dense(1, activation='sigmoid')
    ])
    m.compile(optimizer=Adam(learning_rate=learning_rate),
              loss='binary_crossentropy')
    return m

param_dist = {
    'units':         [16, 32, 64, 128],
    'dropout_rate':  [0.0, 0.2, 0.1],
    'learning_rate': [1e-2, 1e-3],
    'epochs':        [20, 50, 100],
    'batch_size':    [32, 64, 128],
}

best_score, best_params = -1, {}
results = []
n_iter = 20

for i in range(n_iter):
    sample = {k: random.choice(v) for k, v in param_dist.items()}

    clf = KerasClassifier(
        model=build_rnn_model,
        units=sample['units'],
        dropout_rate=sample['dropout_rate'],
        learning_rate=sample['learning_rate'],
        epochs=sample['epochs'],
        batch_size=sample['batch_size'],
        verbose=0,
        random_state=42
    )

    X_train_3d = reshape(X_tune_proc)
    clf.fit(X_train_3d, y_tune)

    X_val_3d    = reshape(X_val_proc)
    y_val_pred  = clf.predict(X_val_3d)

    score = f1_score(y_val, y_val_pred, pos_label=1)
    cm    = confusion_matrix(y_val, y_val_pred)

    print(f"\nTrial {i+1}/{n_iter}: {sample}")
    print("Confusion matrix:\n", cm)
    print(f"F1 = {score:.4f}")

    results.append({**sample, 'f1_val': score})
    if score > best_score:
        best_score, best_params = score, sample

print("\n=== Randomized RNN Search Complete ===")
print("Best F1:", best_score)
print("Best params:", best_params)



Trial 1/20: {'units': 16, 'dropout_rate': 0.0, 'learning_rate': 0.01, 'epochs': 100, 'batch_size': 32}
Confusion matrix:
 [[7776   22]
 [ 100  102]]
F1 = 0.6258

Trial 2/20: {'units': 16, 'dropout_rate': 0.1, 'learning_rate': 0.01, 'epochs': 20, 'batch_size': 64}
Confusion matrix:
 [[7792    6]
 [ 122   80]]
F1 = 0.5556

Trial 3/20: {'units': 32, 'dropout_rate': 0.1, 'learning_rate': 0.001, 'epochs': 100, 'batch_size': 128}
Confusion matrix:
 [[7787   11]
 [ 111   91]]
F1 = 0.5987

Trial 4/20: {'units': 16, 'dropout_rate': 0.0, 'learning_rate': 0.001, 'epochs': 50, 'batch_size': 128}
Confusion matrix:
 [[7773   25]
 [ 100  102]]
F1 = 0.6201

Trial 5/20: {'units': 32, 'dropout_rate': 0.0, 'learning_rate': 0.01, 'epochs': 20, 'batch_size': 64}
Confusion matrix:
 [[7778   20]
 [ 100  102]]
F1 = 0.6296

Trial 6/20: {'units': 128, 'dropout_rate': 0.1, 'learning_rate': 0.001, 'epochs': 50, 'batch_size': 64}
Confusion matrix:
 [[7789    9]
 [ 119   83]]
F1 = 0.5646

Trial 7/20: {'units': 128

In [ ]:
df_results_rnn = pd.DataFrame(results)
print(df_results_rnn.sort_values('f1_val', ascending=False))

    units  dropout_rate  learning_rate  epochs  batch_size    f1_val
12     64           0.0          0.001     100          64  0.634441
11    128           0.0          0.001     100          64  0.634146
4      32           0.0          0.010      20          64  0.629630
0      16           0.0          0.010     100          32  0.625767
3      16           0.0          0.001      50         128  0.620061
13    128           0.0          0.010     100          32  0.617737
2      32           0.1          0.001     100         128  0.598684
6     128           0.1          0.001     100          32  0.583893
5     128           0.1          0.001      50          64  0.564626
14    128           0.1          0.001      50          64  0.564626
1      16           0.1          0.010      20          64  0.555556
17     32           0.1          0.010      20          64  0.553633
10     32           0.2          0.001      20         128  0.365079
15     16           0.1          0

In [ ]:
best_params = {
    'units': 32,
    'dropout_rate': 0.1,
    'learning_rate': 0.001,
    'epochs': 100,
    'batch_size': 128
}
print("\n=== Randomized RNN Search Complete ===")
print("Best params:", best_params)


=== Randomized RNN Search Complete ===
Best params: {'units': 32, 'dropout_rate': 0.1, 'learning_rate': 0.001, 'epochs': 100, 'batch_size': 128}


In [ ]:
pd.concat([X_sub, y_sub.rename('Default_bin')], axis=1)\
  .to_csv("subset_train_40000forXGBMLPRNN.csv", index=False)

In [ ]:
best_params_xgb

{'n_estimators': 500, 'max_depth': 10, 'learning_rate': 0.2}

In [ ]:
n_neg_full = (y_train_full == 0).sum()
n_pos_full = (y_train_full == 1).sum()
scale_pos_weight_full = n_neg_full / n_pos_full

# 10. Train final models on full 2000 data
# -- XGB
final_xgb = Pipeline([
    ('pre', preprocessor),
    ('clf', XGBClassifier(
        use_label_encoder=False, eval_metric='logloss', random_state=42,
        scale_pos_weight=scale_pos_weight_full,
        **best_params_xgb
    ))
])
final_xgb.fit(X_train_full, y_train_full)

c:\Users\z8360\anaconda3\envs\tf_env\lib\site-packages\xgboost\core.py:158: UserWarning: [06:07:08] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['ORIG_AMT', 'ORIG_TRM',
                                                   'NUM_BO', 'DTI',
                                                   'reperformance_flag',
                                                   'Loan.Age', 'FicoBkt',
                                                   'LAST_VAL', 'MLTV']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['PROP_TYP'])])),
                ('clf',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None,...
                               feature_types=None, gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.2,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=10, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=500, n_jobs=None,
                               num_parallel_tree=None, random_state=42, ...))])

In [ ]:
evaluate(final_xgb, X_test, y_test_orig)


=== clf ===
Confusion matrix (true [-1,0] vs pred [-1,0]):
 [[702964   7307]
 [ 17017   5754]]
Accuracy:       0.9668177266786896
Precision(0):   0.4405481969221346
Recall(0):      0.2526898247771288
Specificity:    0.9897123773883489
F1(0):          0.32116543871399866


In [ ]:
best_params_mlp

{'hidden_layer_sizes': (20,), 'alpha': 0.001, 'learning_rate_init': 0.01}

In [ ]:
# -- MLP
final_mlp = Pipeline([
    ('pre', preprocessor),
    ('clf', MLPClassifier(
        max_iter=200, random_state=42, **best_params_mlp
    ))
])
final_mlp.fit(X_train_full, y_train_full)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['ORIG_AMT', 'ORIG_TRM',
                                                   'NUM_BO', 'DTI',
                                                   'reperformance_flag',
                                                   'Loan.Age', 'FicoBkt',
                                                   'LAST_VAL', 'MLTV']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['PROP_TYP'])])),
                ('clf',
                 MLPClassifier(alpha=0.001, hidden_layer_sizes=(20,),
                               learning_rate_init=0.01, random_state=42))])

In [ ]:
best_params

{'units': 32,
 'dropout_rate': 0.1,
 'learning_rate': 0.001,
 'epochs': 100,
 'batch_size': 128}

In [ ]:
best_params_rnn = best_params.copy()

In [ ]:
# -- RNN
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
try:
    from scikeras.wrappers import KerasClassifier
    print("Using KerasClassifier from scikeras.wrappers")
except ImportError:
    try:
        from tensorflow.keras.wrappers.scikit_learn import KerasClassifier
        print("Using KerasClassifier from tensorflow.keras.wrappers.scikit_learn")
    except ImportError:
        print("ERROR: KerasClassifier wrapper not found. Please install scikeras (`pip install scikeras`).")
        KerasClassifier = None
from sklearn.utils import class_weight
from sklearn.metrics import confusion_matrix
import time
import os
import random
import sys

SEED = 42
os.environ['PYTHONHASHSEED']=str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("--- Running Final RNN Training/Evaluation (User's Structure - Fixed Again) ---")

if KerasClassifier is None: raise ImportError("KerasClassifier not found.")

# 1. Instantiate KerasClassifier Wrapper
print("Instantiating KerasClassifier...")
final_keras = KerasClassifier(
    model=build_rnn_model,
    units=best_params_rnn['units'],
    dropout_rate=best_params_rnn['dropout_rate'],
    learning_rate=best_params_rnn['learning_rate'],
    epochs=best_params_rnn['epochs'],
    batch_size=best_params_rnn['batch_size'],
    verbose=0,

)

print("Preprocessing and reshaping training data...")
X_train_3d = preprocessor.transform(X_train_full).reshape(-1, 1, n_features)

# 3. Calculate class_weights_dict
print("Calculating class weights for final training...")
try:
    class_weights_list_final = class_weight.compute_class_weight(
        'balanced', classes=np.unique(y_train_full), y=y_train_full
    )
    class_weights_dict = dict(enumerate(class_weights_list_final))
    print(f"Calculated Class Weights (from y_train_full): {class_weights_dict}")
except Exception as e:
     print(f"Warning: Could not calculate class weights: {e}. Proceeding without them.")
     class_weights_dict = None


print(f"Fitting final_keras model for {best_params_rnn['epochs']} epochs...")
try:
    final_keras.fit(
        X_train_3d,
        y_train_full,
        class_weight=class_weights_dict
    )
    print("Model fitting complete.")
except Exception as e:
    print(f"An error occurred during final_keras.fit: {e}")
    print("Stopping script because fitting failed.")
    raise


#  Preprocess and Reshape Test Data
print("Preprocessing and reshaping test data...")
X_test_3d = preprocessor.transform(X_test).reshape(-1, 1, n_features)

#  Define the Evaluation Function
def evaluate_rnn(model, X, y_orig, n_features):
    """
    Evaluates RNN using the user's specific structure,
    corrected to use predict_proba for consistent evaluation.
    """
    print("\n=== RNN Evaluation (User's Function - Revised) ===")
    try:
        print("  Preprocessing/reshaping data inside evaluate_rnn...")
        X_proc = preprocessor.transform(X).reshape(-1, 1, n_features)
        print(f"  Processed data shape: {X_proc.shape}")

        print("  Generating probabilities using model.predict_proba...")
        if not hasattr(model, 'predict_proba'):
             print("  ERROR: Model object does not have 'predict_proba' method!")
             y_prob_pos = model.predict(X_proc).ravel()
             print("  Warning: predict_proba not found, falling back to predict.")
        else:
            y_prob = model.predict_proba(X_proc)
            if y_prob.shape[1] != 2:
                print(f"  Warning: predict_proba output shape is {y_prob.shape}. Adjusting if possible.")
                if y_prob.shape[1] == 1: y_prob_pos = y_prob.ravel()
                else: raise ValueError(f"Cannot determine positive class probability from shape {y_prob.shape}")
            else:
                y_prob_pos = y_prob[:, 1]

        threshold = 0.5
        print(f"  Applying threshold {threshold} to probabilities...")
        y_bin = (y_prob_pos >= threshold).astype(int)

        y_pred = np.where(y_bin == 1, 0, -1)

        print("  Calculating confusion matrix...")
        cm = confusion_matrix(y_orig, y_pred, labels=[-1, 0])
        tn, fp, fn, tp = cm.ravel()

        print("Confusion matrix (true [-1,0] vs pred [-1,0]):\n", cm)
        total = tp + tn + fp + fn
        accuracy = (tp + tn) / total if total > 0 else 0
        precision_0 = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall_0 = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity_neg1 = tn / (tn + fp) if (tn + fp) > 0 else 0
        f1_0 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0

        print(f"Accuracy:      {accuracy:.4f}")
        print(f"Precision(0):  {precision_0:.4f}")
        print(f"Recall(0):     {recall_0:.4f}")
        print(f"Specificity:   {specificity_neg1:.4f}")
        print(f"F1(0):         {f1_0:.4f}")

    except AttributeError as ae: print(f"  ERROR evaluating: Attribute error - maybe preprocessor not found? {ae}")
    except NotFittedError as nfe: print(f"  ERROR evaluating: Model is not fitted. {nfe}")
    except Exception as e: print(f"  An unexpected error occurred during evaluation: {e}")

print("\nCalling evaluate_rnn function (Revised Version)...")
evaluate_rnn(final_keras, X_test, y_test_orig, n_features)


Using KerasClassifier from scikeras.wrappers
--- Running Final RNN Training/Evaluation (User's Structure - Fixed Again) ---
Instantiating KerasClassifier...
Preprocessing and reshaping training data...
Calculating class weights for final training...
Calculated Class Weights (from y_train_full): {0: 0.5134763870262106, 1: 19.05096618357488}
Fitting final_keras model for 100 epochs...
Model fitting complete.
Preprocessing and reshaping test data...

Calling evaluate_rnn function (Revised Version)...

=== RNN Evaluation (User's Function - Revised) ===
  Preprocessing/reshaping data inside evaluate_rnn...
  Processed data shape: (733042, 1, 11)
  Generating probabilities using model.predict_proba...
  Applying threshold 0.5 to probabilities...
  Calculating confusion matrix...
Confusion matrix (true [-1,0] vs pred [-1,0]):
 [[649833  60438]
 [  2538  20233]]
Accuracy:      0.9141
Precision(0):  0.2508
Recall(0):     0.8885
Specificity:   0.9149
F1(0):         0.3912

--- Script execution f

In [ ]:
evaluate(final_xgb, X_test, y_test_orig)



=== clf ===
Confusion matrix (true [-1,0] vs pred [-1,0]):
 [[702976   7295]
 [ 17137   5634]]
Accuracy:       0.9666703954207262
Precision(0):   0.4357645602908191
Recall(0):      0.24741996398928462
Specificity:    0.9897292723481601
F1(0):          0.31563025210084034


In [ ]:
evaluate(final_mlp, X_test, y_test_orig)


=== clf ===
Confusion matrix (true [-1,0] vs pred [-1,0]):
 [[707802   2469]
 [ 18970   3801]]
Accuracy:       0.9707533811159524
Precision(0):   0.6062200956937799
Recall(0):      0.16692284045496464
Specificity:    0.9965238620188632
F1(0):          0.2617678454598671
